# Compare Chroma, Qdrant and FAISS

The brief names all three as local vector-store examples. Here the same 30 recipe texts, embeddings, normalized vectors, ten questions and top-k=5 are supplied to each engine. Chroma and Qdrant use cosine search; FAISS uses inner product on normalized vectors, making its score equivalent to cosine. We use in-memory stores to avoid modifying the MVP index. FAISS is a similarity-search library: applications must manage text/metadata and persistence around it; Chroma and Qdrant provide collections and metadata support. All have local persistence options, but this test does not measure them.

Recipe hit@5 is only a small retrieval proxy. Index build and search timings exclude embedding generation, are taken in one process on a tiny dataset, and should not be extrapolated to production. Tie-breaking may differ. If a package is missing, install the optional comparison requirements in the project environment.


In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import chromadb
import faiss
from qdrant_client import QdrantClient, models
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

HERE=Path.cwd().resolve()
if HERE.name != 'comparisons_v1': raise RuntimeError('Start this notebook in v2/comparisons_v1')
V2=HERE.parent
sys.path.insert(0,str(V2))
from chef.retrieval import read_recipes
recipes=read_recipes()
cases=json.loads((V2/'data'/'evaluation_v2.json').read_text(encoding='utf-8'))[:10]
targets=['Tomato pasta','Mushroom pasta','Lentil soup','Chickpea salad','Carrot soup','Tofu stir fry','Baked potatoes','Pea soup','Chicken rice bowl','Lentil soup']
texts=[]
for r in recipes:
    items=', '.join(f"{i['quantity']} {i['unit']} {i['name']}" for i in r['ingredients'])
    texts.append(f"Recipe: {r['name']}. {r['overview']} Ingredients: {items}. Instructions: {r['instructions']} Dietary tags: {', '.join(r['dietary_tags'])}.")
embed=HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=str(V2.parent/'storage'/'models'))
vectors=np.asarray(embed.get_text_embedding_batch(texts),dtype=np.float32)
queries=np.asarray(embed.get_text_embedding_batch([x['question'] for x in cases]),dtype=np.float32)
vectors/=np.maximum(np.linalg.norm(vectors,axis=1,keepdims=True),1e-12)
queries/=np.maximum(np.linalg.norm(queries,axis=1,keepdims=True),1e-12)
ids=[str(i) for i in range(len(recipes))]

start=time.perf_counter()
chroma=chromadb.EphemeralClient().create_collection('recipe_compare', metadata={'hnsw:space':'cosine'}, embedding_function=None)
chroma.add(ids=ids, embeddings=vectors.tolist(), documents=texts)
chroma_build=time.perf_counter()-start
start=time.perf_counter()
qdrant=QdrantClient(':memory:')
qdrant.create_collection('recipes',vectors_config=models.VectorParams(size=vectors.shape[1],distance=models.Distance.COSINE))
qdrant.upsert('recipes',points=[models.PointStruct(id=i,vector=vectors[i].tolist()) for i in range(len(recipes))])
qdrant_build=time.perf_counter()-start
start=time.perf_counter()
index=faiss.IndexFlatIP(vectors.shape[1]); index.add(vectors)
faiss_build=time.perf_counter()-start

ranks={}; timing={}
start=time.perf_counter()
for q in queries:
    ranks.setdefault('Chroma',[]).append([int(x) for x in chroma.query(query_embeddings=[q.tolist()],n_results=5)['ids'][0]])
timing['Chroma']=time.perf_counter()-start
start=time.perf_counter()
for q in queries:
    ranks.setdefault('Qdrant',[]).append([int(x.id) for x in qdrant.query_points('recipes',query=q.tolist(),limit=5).points])
timing['Qdrant']=time.perf_counter()-start
start=time.perf_counter()
for q in queries:
    _, indices=index.search(q.reshape(1,-1),5)
    ranks.setdefault('FAISS',[]).append(indices[0].tolist())
timing['FAISS']=time.perf_counter()-start
build={'Chroma':chroma_build,'Qdrant':qdrant_build,'FAISS':faiss_build}
rows=[]
for name, rankings in ranks.items():
    hits=[target in [recipes[i]['name'] for i in rank] for target,rank in zip(targets,rankings)]
    rows.append({'backend':name,'recipe_hit_at_5':float(np.mean(hits)), 'index_build_ms':round(build[name]*1000,2), 'mean_query_ms':round(timing[name]*1000/len(cases),3)})
display(pd.DataFrame(rows))
comparison=pd.DataFrame([{'id':case['id'], 'expected':target, **{backend:[recipes[i]['name'] for i in ranking[j]] for backend,ranking in ranks.items()}} for j,(case,target) in enumerate(zip(cases,targets))])
display(comparison)


## What this does and does not establish

The same embeddings should yield similar neighbors, subject to index algorithms and tie handling. Search latency here is dominated by tiny-test overhead and is not a fair assessment of persistent databases, larger collections, filtered search or concurrent traffic. Compare those separately before selecting a production backend. Exact distance and score semantics can vary; do not compare raw scores without checking them.
